# CSIRO Biomass - Improved Inference with TTA

**改良点**:
1. Test Time Augmentation (TTA) - 5種類の変換
2. EMAモデルとSWAモデルの両方をサポート
3. 改良されたアンサンブル戦略

In [ ]:
# Kaggle環境用: timmのインストール
!pip uninstall -y timm -q
!pip install -q --no-deps /kaggle/input/wheels-csiro/timm-1.0.22-py3-none-any.whl

In [ ]:
import os
import gc
import random
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from tqdm import tqdm

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# T4×2対応: GPU数を確認
n_gpus = torch.cuda.device_count()
print(f"\n{'='*60}")
print(f"🖥️ GPU Configuration")
print(f"{'='*60}")
print(f"Available GPUs: {n_gpus}")

for i in range(n_gpus):
    gpu_name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1024**3
    free = torch.cuda.mem_get_info(i)[0] / 1024**3
    print(f"GPU {i}: {gpu_name} | Total: {vram:.1f}GB | Free: {free:.1f}GB")

# デバイス設定
device0 = torch.device("cuda:0" if n_gpus > 0 else "cpu")
device1 = torch.device("cuda:1" if n_gpus > 1 else device0)

print(f"\nPrimary device: {device0}")
print(f"Secondary device: {device1}")
print(f"PyTorch: {torch.__version__}")
print(f"timm: {timm.__version__}")
print(f"{'='*60}")

In [ ]:
class CFG:
    TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
    DATA_DIR = Path("/kaggle/input/csiro-biomass")
    MODEL_DIR = Path("/kaggle/input/csiro-improved-models")  # 改良版モデルのパス
    
    # T4メモリ制約に対応した設定
    IMG_SIZE = 448  # 512から縮小（T4メモリ節約）
    N_FOLDS = 5
    BACKBONE = "vit_huge_plus_patch16_dinov3.lvd1689m"
    
    BATCH_SIZE = 1
    NUM_WORKERS = 2  # T4×2で並列処理
    
    # TTA設定（メモリ節約のため制限）
    TTA_ENABLED = True
    TTA_TRANSFORMS = ["original", "hflip"]  # 5種類から2種類に削減
    
    # モデルタイプ設定（メモリ節約）
    USE_EMA = True  # EMAモデルを使用
    USE_SWA = False  # SWAは使用しない（メモリ節約）
    
    # アンサンブル重み
    FOLD_WEIGHTS = [1.0, 0.9, 0.95, 1.1, 0.95]  # Fold毎の重み
    
    # T4×2用のGPU割り当て
    GPU0_FOLDS = [0, 2, 4]  # GPU0で処理するFold
    GPU1_FOLDS = [1, 3]     # GPU1で処理するFold
    
    # メモリ最適化設定
    CLEAR_CACHE_INTERVAL = 20  # 何画像ごとにキャッシュをクリアするか
    USE_FP16 = True  # FP16推論を使用

In [ ]:
# データ読み込み
test_df = pd.read_csv(CFG.DATA_DIR / "test.csv")
print(f"test_df columns: {test_df.columns.tolist()}")
print(f"test_df shape: {test_df.shape}")
print(f"\nFirst 10 rows:")
print(test_df.head(10))

# ユニークなテスト画像を取得
test_wide = test_df[["image_path"]].drop_duplicates().reset_index(drop=True)
print(f"\nUnique test images: {len(test_wide)}")

In [ ]:
# ============================================================
# MODEL DEFINITION (improved_training.pyと同じ)
# ============================================================

class LocalMambaBlock(nn.Module):
    """改良版トレーニングコードと同じ実装"""
    def __init__(self, dim, kernel_size=5, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.dwconv = nn.Conv1d(dim, dim, kernel_size=kernel_size, padding=kernel_size//2, groups=dim)
        self.gate = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        shortcut = x
        x = self.norm(x)
        x = x * torch.sigmoid(self.gate(x))
        x = self.dwconv(x.transpose(1, 2)).transpose(1, 2)
        x = self.proj(x)
        return shortcut + self.drop(x)


class BiomassModel(nn.Module):
    """改良版トレーニングコードと同じ実装"""
    def __init__(self, model_name, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool="")
        nf = self.backbone.num_features
        
        # 推論時はGradient Checkpointingは不要だが、構造の一貫性のため
        if hasattr(self.backbone, "set_grad_checkpointing"):
            self.backbone.set_grad_checkpointing(False)  # 推論時はOFF
        
        self.fusion = nn.Sequential(
            LocalMambaBlock(nf, kernel_size=5, dropout=0.1),
            LocalMambaBlock(nf, kernel_size=5, dropout=0.1)
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        self.head_green = nn.Sequential(
            nn.Linear(nf, nf//2), nn.GELU(), nn.Dropout(0.2), 
            nn.Linear(nf//2, 1), nn.Softplus()
        )
        self.head_dead = nn.Sequential(
            nn.Linear(nf, nf//2), nn.GELU(), nn.Dropout(0.2), 
            nn.Linear(nf//2, 1), nn.Softplus()
        )
        self.head_clover = nn.Sequential(
            nn.Linear(nf, nf//2), nn.GELU(), nn.Dropout(0.2), 
            nn.Linear(nf//2, 1), nn.Softplus()
        )

    def forward(self, x):
        left, right = x
        x_l = self.backbone(left)
        x_r = self.backbone(right)
        x = self.fusion(torch.cat([x_l, x_r], dim=1))
        x = self.pool(x.transpose(1, 2)).flatten(1)
        green = self.head_green(x)
        dead = self.head_dead(x)
        clover = self.head_clover(x)
        gdm = green + clover
        total = green + clover + dead
        return torch.cat([green, dead, clover, gdm, total], dim=1)

print("✅ Model defined (improved version)")

In [ ]:
# ============================================================
# TTA FUNCTIONS
# ============================================================

def apply_tta_transform(img, transform_name):
    """Test Time Augmentation変換を適用"""
    if transform_name == "original":
        return img
    elif transform_name == "hflip":
        return torch.flip(img, [-1])  # 水平反転
    elif transform_name == "vflip":
        return torch.flip(img, [-2])  # 垂直反転
    elif transform_name == "rotate90":
        return torch.rot90(img, k=1, dims=[-2, -1])  # 90度回転
    elif transform_name == "rotate270":
        return torch.rot90(img, k=3, dims=[-2, -1])  # 270度回転
    else:
        return img


@torch.no_grad()
def predict_with_tta(model, left, right, device, tta_transforms=None):
    """Test Time Augmentationで予測"""
    if tta_transforms is None or not CFG.TTA_ENABLED:
        tta_transforms = ["original"]
    
    predictions = []
    
    for transform_name in tta_transforms:
        # 左右の画像に同じ変換を適用
        left_aug = apply_tta_transform(left, transform_name)
        right_aug = apply_tta_transform(right, transform_name)
        
        # 予測
        with torch.cuda.amp.autocast():
            pred = model((left_aug, right_aug))
        
        predictions.append(pred)
    
    # 予測を平均化
    final_pred = torch.mean(torch.stack(predictions), dim=0)
    return final_pred

print("✅ TTA functions defined")

In [ ]:
# ============================================================
# DATASET
# ============================================================

class TestDataset(Dataset):
    def __init__(self, df, data_dir, transform):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.data_dir / row["image_path"]
        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        left = img.crop((0, 0, w // 2, h))
        right = img.crop((w // 2, 0, w, h))
        left = self.transform(left)
        right = self.transform(right)
        return left, right, row["image_path"]


def collate_fn(batch):
    lefts = torch.stack([b[0] for b in batch])
    rights = torch.stack([b[1] for b in batch])
    paths = [b[2] for b in batch]
    return lefts, rights, paths


# Transform定義
test_tfms = T.Compose([
    T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Dataset defined")

In [ ]:
# ============================================================
# MAIN INFERENCE WITH TTA (T4×2 Optimized)
# ============================================================

print("="*60)
print("🚀 CSIRO Biomass Inference - T4×2 Optimized Version")
print("="*60)
print(f"Image Size: {CFG.IMG_SIZE}px (optimized for T4)")
print(f"TTA Enabled: {CFG.TTA_ENABLED}")
if CFG.TTA_ENABLED:
    print(f"TTA Transforms: {CFG.TTA_TRANSFORMS} (reduced for memory)")
print(f"Use EMA: {CFG.USE_EMA}")
print(f"Use SWA: {CFG.USE_SWA}")
print(f"FP16 Inference: {CFG.USE_FP16}")
if n_gpus > 1:
    print(f"GPU0 Folds: {CFG.GPU0_FOLDS}")
    print(f"GPU1 Folds: {CFG.GPU1_FOLDS}")
print("="*60)

# DataLoader準備
test_dataset = TestDataset(test_wide, CFG.DATA_DIR, test_tfms)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True  # T4×2では有効化
)

print(f"\n📊 DataLoader: {len(test_dataset)} images")

In [ ]:
# ============================================================
# T4×2 OPTIMIZED INFERENCE
# ============================================================

def inference_on_device(fold, device, test_loader, model_type="ema"):
    """単一GPU上で特定のFoldを推論"""
    
    # モデルパス設定
    if model_type == "ema":
        model_path = CFG.MODEL_DIR / f"best_ema_fold{fold}.pth"
    else:
        model_path = CFG.MODEL_DIR / f"best_swa_fold{fold}.pth"
    
    if not model_path.exists():
        # EMAがない場合、通常モデルを試す
        model_path = CFG.MODEL_DIR / f"best_model_fold{fold}.pth"
        if not model_path.exists():
            print(f"Fold {fold}: Model not found")
            return None
    
    print(f"\n{'='*40}")
    print(f"Fold {fold} - {model_type.upper()} on {device}")
    print(f"{'='*40}")
    
    # デバイス固有のメモリクリア
    if device.type == 'cuda':
        with torch.cuda.device(device):
            torch.cuda.empty_cache()
    
    # モデルロード（CPUで最初にロード）
    model = BiomassModel(CFG.BACKBONE, pretrained=False)
    state_dict = torch.load(model_path, map_location="cpu")
    
    # DataParallelラッパー対応
    if list(state_dict.keys())[0].startswith("module."):
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    
    # メモリ使用量表示
    if device.type == 'cuda':
        with torch.cuda.device(device):
            allocated = torch.cuda.memory_allocated() / 1024**3
            print(f"Memory allocated on {device}: {allocated:.1f} GB")
    
    # 予測
    preds = []
    paths = []
    
    with torch.no_grad():
        for idx, (left, right, p) in enumerate(tqdm(test_loader, desc=f"Fold {fold}")):
            left = left.to(device)
            right = right.to(device)
            
            # FP16推論（メモリ節約）
            if CFG.USE_FP16:
                with torch.cuda.amp.autocast():
                    if CFG.TTA_ENABLED:
                        pred = predict_with_tta(model, left, right, device, CFG.TTA_TRANSFORMS)
                    else:
                        pred = model((left, right))
            else:
                if CFG.TTA_ENABLED:
                    pred = predict_with_tta(model, left, right, device, CFG.TTA_TRANSFORMS)
                else:
                    pred = model((left, right))
            
            preds.append(pred.cpu().numpy())
            paths.extend(p)
            
            # 定期的なメモリクリア
            if (idx + 1) % CFG.CLEAR_CACHE_INTERVAL == 0:
                if device.type == 'cuda':
                    with torch.cuda.device(device):
                        torch.cuda.empty_cache()
    
    preds = np.vstack(preds)
    print(f"Fold {fold} predictions shape: {preds.shape}")
    
    # モデル削除とメモリクリア
    del model, state_dict
    if device.type == 'cuda':
        with torch.cuda.device(device):
            torch.cuda.empty_cache()
    gc.collect()
    
    return preds, paths


# 並列推論の実行
print(f"\n{'='*60}")
print("Starting parallel inference on T4×2")
print(f"{'='*60}")

all_predictions = {}
all_paths = None

# GPU0での処理
print(f"\n🎮 GPU 0: Processing folds {CFG.GPU0_FOLDS}")
for fold in CFG.GPU0_FOLDS:
    preds, paths = inference_on_device(fold, device0, test_loader, "ema")
    if preds is not None:
        all_predictions[fold] = preds * CFG.FOLD_WEIGHTS[fold]
        if all_paths is None:
            all_paths = paths

# GPU1での処理（2つ目のGPUがある場合）
if n_gpus > 1:
    print(f"\n🎮 GPU 1: Processing folds {CFG.GPU1_FOLDS}")
    for fold in CFG.GPU1_FOLDS:
        preds, paths = inference_on_device(fold, device1, test_loader, "ema")
        if preds is not None:
            all_predictions[fold] = preds * CFG.FOLD_WEIGHTS[fold]
else:
    # 1GPUしかない場合は同じGPUで処理
    print(f"\n⚠️ Single GPU detected, processing remaining folds on GPU0")
    for fold in CFG.GPU1_FOLDS:
        preds, paths = inference_on_device(fold, device0, test_loader, "ema")
        if preds is not None:
            all_predictions[fold] = preds * CFG.FOLD_WEIGHTS[fold]

# 予測を順番通りに並べ替え
sorted_predictions = []
total_weight = 0
for fold in range(CFG.N_FOLDS):
    if fold in all_predictions:
        sorted_predictions.append(all_predictions[fold])
        total_weight += CFG.FOLD_WEIGHTS[fold]

print(f"\n{'='*40}")
print(f"Total models used: {len(sorted_predictions)}")
print(f"{'='*40}")

In [ ]:
# 最終アンサンブル
print(f"\n{'='*40}")
print(f"Creating Final Ensemble")
print(f"{'='*40}")

# 重み付き平均でアンサンブル
ensemble = np.sum(sorted_predictions, axis=0) / total_weight
print(f"Ensemble shape: {ensemble.shape}")
print(f"Models in ensemble: {len(sorted_predictions)}")

# メモリ状況の最終確認
if torch.cuda.is_available():
    for i in range(n_gpus):
        with torch.cuda.device(i):
            allocated = torch.cuda.memory_allocated() / 1024**3
            free = torch.cuda.mem_get_info()[0] / 1024**3
            print(f"GPU {i} - Allocated: {allocated:.1f}GB, Free: {free:.1f}GB")

# Wide形式のDataFrame作成
preds_wide = pd.DataFrame(ensemble, columns=CFG.TARGETS)
preds_wide.insert(0, 'image_path', all_paths)

print(f"\nPredictions summary:")
print(preds_wide[CFG.TARGETS].describe())

# Long形式に変換
preds_long = preds_wide.melt(
    id_vars=['image_path'],
    value_vars=CFG.TARGETS,
    var_name='target_name',
    value_name='target'
)

print(f"\nPredictions long format:")
print(preds_long.head(10))

In [ ]:
# test_dfとマージしてsample_idを取得
submission = pd.merge(
    test_df[['sample_id', 'image_path', 'target_name']],
    preds_long,
    on=['image_path', 'target_name'],
    how='left'
)

# 必要なカラムのみ保持
submission = submission[['sample_id', 'target']]

# 欠損値チェック
missing_count = submission['target'].isna().sum()
if missing_count > 0:
    print(f"\n⚠️ Warning: {missing_count} missing predictions!")
    submission['target'] = submission['target'].fillna(0.0)
else:
    print("\n✅ All predictions matched!")

# sample_idでソート、負の値をクリップ
submission = submission.sort_values('sample_id').reset_index(drop=True)
submission['target'] = submission['target'].clip(lower=0)

# 保存
submission.to_csv("submission_improved.csv", index=False)

print(f"\n✅ Saved: submission_improved.csv")
print(f"Shape: {submission.shape}")
print(f"\nFirst 10 rows:")
print(submission.head(10))
print(f"\nTarget statistics:")
print(submission['target'].describe())

In [ ]:
# 検証
print(f"\n" + "="*50)
print("VALIDATION")
print("="*50)
print(f"Expected rows: {len(test_df)}")
print(f"Actual rows: {len(submission)}")
print(f"Match: {len(submission) == len(test_df)}")
print(f"No missing: {not submission['target'].isna().any()}")
print(f"All positive: {(submission['target'] >= 0).all()}")
print(f"Unique IDs: {submission['sample_id'].is_unique}")

print("\n🎉 Inference complete!")